In [1]:
from dotenv import load_dotenv

_ = load_dotenv()

In [2]:
# automatically reload all modules before executing new code. The captures changes in
# local packages.
%load_ext autoreload
%autoreload 2

## Step 1: Create Sub Agents

In [8]:
%%writefile ../src/deep_agents_from_scratch/task_tool.py
"""Task delegation tools for context isolation through sub-agents.

This module provides the core infrastructure for creating and managing sub-agents
with isolated contexts. Sub-agents prevent context clash by operating with clean
context windows containing only their specific task description.
"""

from typing import Annotated, NotRequired
from typing_extensions import TypedDict

from langchain.messages import ToolMessage
from langchain.tools import BaseTool, InjectedToolCallId, tool
from langgraph.prebuilt import InjectedState
from langchain.agents import create_agent

from langgraph.types import Command

from deep_agents_from_scrath.prompts import TASK_DESCRIPTION_PREFIX
from deep_agents_from_scrath.state import DeepAgentState

class SubAgent(TypedDict):
    """Configuration for a specialized sub-agent."""

    name: str
    description: str
    prompt: str
    tools: NotRequired[list[str]]
    

def _create_task_tool(tools, subagents: list[SubAgent], model, state_schema):
    """Create a task delegation tool that enables context isolation through sub-agents.
    
    This function implements the core pattern for spawning specialized sub-agents with
    isolated contexts, preventing context clash and confusion in complex multi-step tasks.
    
    Args:
        tools: List of available tools that can be assigned to sub-agents
        subagents: List of specialized sub-agent configurations
        model: The language model to use for all agents
        state_schema: The state schema (typically DeepAgentState)
        
    Returns:
        A 'task' tool that can delegate work to specialized sub-agents
    """
    
    # create agent registry
    agents = {}
    
    # build tool name mapping for selective tool assignment
    tools_by_name = {}
    for tool_ in tools:
        if not isinstance(tool_, BaseTool):
            tool_ = tool(tool_)
        tools_by_name[tool_.name] = tool_
        
    # create specialized sub-agents based on configurations
    for _agent in subagents:
        if "tools" in _agent:
            #use specific tools if specified
            _tools = [tools_by_name[t] for t in _agent["tools"]]
        else:
            # default to all tools
            _tools = tools
        
        agents[_agent["name"]] = create_agent(
            model,
            system_prompt=_agent["prompt"],
            tools=_tools,
            state_schema=state_schema
        )
        
    # generate description of available sub-agents for the tool description
    other_agents_string = [
        f"-{_agent["name"]}: {_agent["description"]}" for _agent in subagents 
    ]

    @tool(description=TASK_DESCRIPTION_PREFIX.format(other_agents=other_agents_string))
    def task(
        description: str,
        subagent_type: str,
        state: Annotated[DeepAgentState, InjectedState],
        tool_call_id: Annotated[str, InjectedToolCallId]
    ):
        """Delegate a task to a specialized sub-agent with isolated context.
        
        This creates a fresh context for the sub-agent containing only the task description,
        preventing context pollution from the parent agent's conversation history.
        """
        
        # validate requested agent types exists
        if subagent_type not in agents:
            return f"Error: invoked agent of type {subagent_type}, the only allowed types are {[f'`{k}' for k in agents]}"
        
        # get the requested sub-agent
        sub_agent = agents[subagent_type]
        
        # create isolated context with only the task description
        # this is the key to context isolation - no parent history
        state["messages"] = [{"role": "user", "content": description}]

        # execute the sub-agent in isolation
        result = sub_agent.invoke(state)
        
        # return results to parent agent via command state update
        return Command(
            update={
                "files": result.get("files", {}), # merge any file changes
                "messages": [
                    # sub-agent result becomes a ToolMessage in parent context
                    ToolMessage(
                        result["messages"][-1].content, tool_call_id=tool_call_id
                    )
                ]
            }
        )

    return task

Overwriting ../src/deep_agents_from_scratch/task_tool.py


In [4]:
from utils import show_prompt

from deep_agents_from_scratch.prompts import SUBAGENT_USAGE_INSTRUCTIONS

show_prompt(SUBAGENT_USAGE_INSTRUCTIONS)

╭─────────────────────────────────── Prompt ───────────────────────────────────╮
│                                                                              │
│  You can delegate tasks to sub-agents.                                       │
│                                                                              │
│  <Task>                                                                      │
│  Your role is to coordinate research by delegating specific research tasks   │
│  to sub-agents.                                                              │
│  </Task>                                                                     │
│                                                                              │
│  <Available Tools>                                                           │
│  1. **task(description, subagent_type)**: Delegate research tasks to         │
│  specialized sub-agents                                                      │
│     - description: Clear, 

In [ ]:
from datetime import datetime

from langchain.chat_models import init_chat_model
from langchain.agents import create_agent

from deep_agents_from_scratch.prompts import SUBAGENT_USAGE_INSTRUCTIONS
from deep_agents_from_scratch.state import DeepAgentState
from deep_agents_from_scratch.task_tool import _create_task_tool

# limits
max_concurrent_research_units = 3
max_researcher_iterations = 3

# mock search result
search_result = """The Model Context Protocol (MCP) is an open standard protocol developed
by Anthropic to enable seamless integration between AI models and external systems like
tools, databases, and other services. It acts as a standardized communication layer,
allowing AI models to access and utilize data from various sources in a consistent and
efficient manner. Essentially, MCP simplifies the process of connecting AI assistants
to external services by providing a unified language for data exchange."""

# mock search tool
@tool(parse_docstring=True)
def web_search(query: str) -> str:
    """Search the web for information on a specific topic.
    
    This tool performs web searches and returns relevant results
    for the given query. Use this when you need to gather information from
    the internet about any topic.
    
    Args:
        query: The search query string. Be specific and clear about what
               information you're looking for.
               
    Returns:
        Search results from the search engine.
        
    Example:
        web_search("machine learning applications in healthcare")
    """
    

# add mock research instructions
SIMPLE_RESEARCH_INSTRUCTIONS = """You are a researcher. Research the topic provided to 
you. IMPORTANT: Just make a single call to the web_search tool and use the result
provided by the tool to answer the provided topic."""

# create research sub-agent
research_sub_agent = {
    "name": "research-agent",
    "description": "Delegate research to the sub-agent researcher. Only give this researcher one topic at a time.",
    "prompt": SIMPLE_RESEARCH_INSTRUCTIONS,
    "tools": ["web_search"]
}

# create agent using create_agent directly
model = init_chat_model(model="anthropic:claude-sonnet-4-20250514", temperature=0.0)

# tools for sub-agent
sub_agent_tools = [web_search]

# create task tool to delegate tasks to sub-agents
task_tool = _create_task_tool(
    sub_agent_tools,
    [research_sub_agent],
    model,
    DeepAgentState
)

# tools
delegation_tools = [task_tool]

# create agent with system prompt
agent = create_agent(
    model,
    delegation_tools,
    system_prompt=SUBAGENT_USAGE_INSTRUCTIONS.format(
        max_concurrent_research_units=max_concurrent_research_units,
        max_researcher_iterations=max_researcher_iterations,
        date=datetime.now().strftime("%a %b %-d, %Y")
    )
)

# show the agent
agent

ModuleNotFoundError: No module named 'deep_agents_from_scrath'